In [11]:
import h3
import json
import pandas as pd
import numpy as np

crowd_mapping_df = pd.read_csv("../data/raw/popular_times/popular_times_malang.csv")
poi_grid_df = pd.read_csv("../data/processed/poi_grid_malang.csv")
venue_grid_df = pd.read_csv("../data/processed/venue_to_grid_mapping.csv")

grouped = venue_grid_df.groupby("grid_id")

rows = []

In [12]:
for grid_id, venues_in_grid in grouped:

    grid_lat = venues_in_grid.iloc[0]["grid_lat"]
    grid_lon = venues_in_grid.iloc[0]["grid_lon"]

    if (venues_in_grid["tier"] == 4).all():
        rows.append({
            "grid_id"      : grid_id,
            "grid_lat"     : grid_lat,
            "grid_lon"     : grid_lon,
            "day_name"     : None,
            "day_int"      : None,
            "hour"         : None,
            "crowd_score"  : None,
            "source_venue" : None,
            "tier"         : 4,
        })
        continue

    merged = venues_in_grid.merge(
        crowd_mapping_df,
        on="venue_name",
        how="inner"
    )

    max_idx = merged.groupby(
        ["day_name", "day_int", "hour"]
    )["busyness_score"].idxmax()
    max_crowd = merged.loc[max_idx].reset_index(drop=True)


    for _, row in max_crowd.iterrows():
        rows.append({
            "grid_id"     : grid_id,
            "grid_lat"    : grid_lat,
            "grid_lon"    : grid_lon,
            "day_name"    : row["day_name"],
            "day_int"     : row["day_int"],
            "hour"        : row["hour"],
            "crowd_score" : row["busyness_score"],
            "source_venue" : row["venue_name"],
            "tier"        : int(venues_in_grid["tier"].min()),
        })

df_crowd_per_grid = pd.DataFrame(rows)
df_crowd_per_grid.to_csv("../data/processed/crowd_per_grid.csv", index=False)

